# 결측치처리

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import numpy as np
import pandas as pd

In [3]:
!ls /content/drive/MyDrive/workspace/lecture/data/

dirty_cafe_sales.csv	Students_Performance_knn.csv
healthcare_dataset.csv	Students_Performance_mv.csv
result.csv		Wine_Quality.csv


In [4]:
# CSV 파일 읽기 및 전처리
df = pd.read_csv("/content/drive/MyDrive/workspace/lecture/data/dirty_cafe_sales.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    10000 non-null  object
 1   Item              9667 non-null   object
 2   Quantity          9862 non-null   object
 3   Price Per Unit    9821 non-null   object
 4   Total Spent       9827 non-null   object
 5   Payment Method    7421 non-null   object
 6   Location          6735 non-null   object
 7   Transaction Date  9841 non-null   object
dtypes: object(8)
memory usage: 625.1+ KB


In [6]:
# 특정 컬럼을 숫자형으로 변환
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')
df['Price Per Unit'] = pd.to_numeric(df['Price Per Unit'], errors='coerce')
df['Total Spent'] = pd.to_numeric(df['Total Spent'], errors='coerce')

print(df.dtypes)
print("\nNaN 개수:")
print(df[['Quantity', 'Price Per Unit', 'Total Spent']].isnull().sum())

Transaction ID       object
Item                 object
Quantity            float64
Price Per Unit      float64
Total Spent         float64
Payment Method       object
Location             object
Transaction Date     object
dtype: object

NaN 개수:
Quantity          479
Price Per Unit    533
Total Spent       502
dtype: int64


In [8]:
# 1. 결측치 확인
print("결측치 처리 실습")

print("[1단계] 결측치 확인")

print("\n각 컬럼별 결측치 개수:")
print(df.isnull().sum()) #-> 각 컬럼별 결측치 개수

print("\n각 컬럼별 결측치 비율:")
missing_ratio = (df.isnull().sum() / len(df) * 100).round(2)
print(missing_ratio)

# 전체 결측치 구하기
total_missing = df.isnull().sum().sum()
print(f"\n전체 결측치 개수: {total_missing}")
total_values = df.shape[0] * df.shape[1]
print(f"전체 결측치 비율: {total_missing/total_values*100:.2f}")

결측치 처리 실습
[1단계] 결측치 확인

각 컬럼별 결측치 개수:
Transaction ID         0
Item                 333
Quantity             479
Price Per Unit       533
Total Spent          502
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64

각 컬럼별 결측치 비율:
Transaction ID       0.00
Item                 3.33
Quantity             4.79
Price Per Unit       5.33
Total Spent          5.02
Payment Method      25.79
Location            32.65
Transaction Date     1.59
dtype: float64

전체 결측치 개수: 7850
전체 결측치 비율: 9.81


In [9]:
# 2. 결측치 처리 방법
# 1) 삭제
df_dropna = df.dropna()

print(f"\n방법 1) 결측치 행 삭제")
print(f"  - 원본 데이터: {len(df)}행")
print(f"  - 삭제 후: {len(df_dropna)}행")
print(f"  - 삭제된 행: {len(df) - len(df_dropna)}행 ({(len(df)-len(df_dropna))/len(df)*100:.1f}%)")


방법 1) 결측치 행 삭제
  - 원본 데이터: 10000행
  - 삭제 후: 4096행
  - 삭제된 행: 5904행 (59.0%)


In [10]:
# 2) 평균값으로 대체 (수치형 컬럼만)
df_fillna_mean = df.copy()
numeric_columns = df.select_dtypes(include=[np.number]).columns

for col in numeric_columns:
  if df[col].isnull().sum() > 0:
    mean_value = df[col].mean()
    df_fillna_mean[col].fillna(mean_value, inplace=True)
    print(f"  - {col}: 평균 {mean_value:.2f}로 {df[col].isnull().sum()}개 대체")

  - Quantity: 평균 3.03로 479개 대체
  - Price Per Unit: 평균 2.95로 533개 대체
  - Total Spent: 평균 8.92로 502개 대체


/tmp/ipython-input-4263119479.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_fillna_mean[col].fillna(mean_value, inplace=True)


In [11]:
# 최종 선택한 데이터
df_final = df_fillna_mean.copy()
print("최종 선택: 평균값 대체 방법")
print(f"처리 완료 후 결측치: {df_final.isnull().sum().sum()}개")

최종 선택: 평균값 대체 방법
처리 완료 후 결측치: 6336개


# 이상치 처리

In [12]:
# 분석할 컬럼
columns = ["Quantity", "Price Per Unit", "Total Spent"]

# 1. 이상치 탐지(IQR 방법)
print("[1단계] IQR 방법으로 이상치 탐지")

outlier_info = {}
for col in columns:
  Q1 = df[col].quantile(0.25)
  Q3 = df[col].quantile(0.75)
  IQR = Q3 - Q1

  lower_bound = Q1 - 1.5 * IQR
  upper_bound = Q3 + 1.5 * IQR

  outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
  # 하한보다 작거나 상한보다 큰 값들을 선정 -> 이상치
  outlier_count = len(outliers)

  outlier_info[col] = {
      "Q1": Q1,
      "Q3": Q3,
      "IQR": IQR,
      "lower_bound": lower_bound,
      "upper_bound": upper_bound,
      "count": outlier_count
  }
  print(f"\n{col}:")
  print(f"  - Q1 (25%): {Q1:.2f}")
  print(f"  - Q3 (75%): {Q3:.2f}")
  print(f"  - IQR: {IQR:.2f}")
  print(f"  - 정상 범위: [{lower_bound:.2f}, {upper_bound:.2f}]")
  print(f"  - 이상치 개수: {outlier_count}개 ({outlier_count/len(df)*100:.1f}%)")

[1단계] IQR 방법으로 이상치 탐지

Quantity:
  - Q1 (25%): 2.00
  - Q3 (75%): 4.00
  - IQR: 2.00
  - 정상 범위: [-1.00, 7.00]
  - 이상치 개수: 0개 (0.0%)

Price Per Unit:
  - Q1 (25%): 2.00
  - Q3 (75%): 4.00
  - IQR: 2.00
  - 정상 범위: [-1.00, 7.00]
  - 이상치 개수: 0개 (0.0%)

Total Spent:
  - Q1 (25%): 4.00
  - Q3 (75%): 12.00
  - IQR: 8.00
  - 정상 범위: [-8.00, 24.00]
  - 이상치 개수: 259개 (2.6%)


In [13]:
print("[2단계] 이상치 처리 - 경계값으로 대체")

df_processed = df.copy()

col = "Total Spent"
lower = outlier_info[col]["lower_bound"]
upper = outlier_info[col]["upper_bound"]

# 원본에서 이상치 개수
outlier_count = outlier_info[col]["count"]

# 경계값으로 대체
df_processed[col] = df_processed[col].clip(lower=lower, upper=upper)

print(f"\t처리 전 평균: {df[col].mean():.2f}")
print(f"\t처리 전 표준편차: {df[col].std():.2f}")
print(f"\t처리 전 최소값: {df[col].min():.2f}")
print(f"\t처리 전 최대값: {df[col].max():.2f}")
print("="*70)
print(f"\t처리 후 평균: {df_processed[col].mean():.2f}")
print(f"\t처리 후 표준편차: {df_processed[col].std():.2f}")
print(f"\t처리 후 최소값: {df_processed[col].min():.2f}")
print(f"\t처리 후 최대값: {df_processed[col].max():.2f}")

[2단계] 이상치 처리 - 경계값으로 대체
	처리 전 평균: 8.92
	처리 전 표준편차: 6.01
	처리 전 최소값: 1.00
	처리 전 최대값: 25.00
	처리 후 평균: 8.90
	처리 후 표준편차: 5.94
	처리 후 최소값: 1.00
	처리 후 최대값: 24.00


# Groupby

In [14]:
# Groupby 로 집계
columns = ['Quantity', 'Price Per Unit', 'Total Spent']

# 카테고리별 평균 계산
grouped_result = df_processed.groupby('Item')[columns].mean()
grouped_result = grouped_result.reset_index()
print(grouped_result)

       Item  Quantity  Price Per Unit  Total Spent
0      Cake  3.045746        3.000000     9.129596
1    Coffee  3.050725        2.000000     6.084305
2    Cookie  2.974013        1.000000     2.966184
3     ERROR  3.083942        2.944444     8.889091
4     Juice  3.000890        3.000000     8.954260
5     Salad  3.022831        5.000000    14.933394
6  Sandwich  3.032710        4.000000    12.165258
7  Smoothie  3.061787        4.000000    12.261719
8       Tea  3.050290        1.500000     4.548991
9   UNKNOWN  2.919003        2.880368     8.541925


In [15]:
# CSV 파일로 저장
save_dir = '/content/drive/MyDrive/workspace/lecture/data/clean-data.csv'

grouped_result.to_csv(save_dir, index=False, encoding='utf-8')
print("데이터 저장 완료")

데이터 저장 완료
